# ISM / Confocal image processing for fluorescence lifetime reconstruction. DFD acquisition method

### Import libraries to process 4D ISM dataset (Nx, Ny, time bin, detector channel) of both acquired sample and IRF  

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import h5py
import matplotlib.pyplot as plt
import numpy as np

import brighteyes_flim.tools_phasor as flim
import brighteyes_ism.analysis.Graph_lib as gr
import brighteyes_ism.dataio.mcs as mcs
from scipy.ndimage import shift
from skimage.registration import phase_cross_correlation


### Load the ISM dataset and align the histograms along the channels' dimension

In [ ]:
#load the data

with h5py.File(r"C:\Users\REPLACE_ME\Desktop\images\data-10-04-2024-17-42-38.h5","r") as f:
    
        print(f.keys())
        data_input = f["data"]  # image with the decay histograms in each pixel realigned 
                                     # with respect to the signal which triggers the beginning of the excitation
        data = data_input[0, 0,...]

In [ ]:
# load the IRF

data_path_irf = r"C:\Users\REPLACE_ME\Desktop\images\data-10-04-2024-18-31-53.h5"
data_irf = h5py.File(data_path_irf)

image_irf = data_irf["data"]
data_hist_irf = np.sum(image_irf, axis=(0, 1, 2, 3))
print(data_hist_irf.shape)

In [ ]:
#load data laser (26th channel) for the convallaria and calculate the correspondent phasor

data_path = r"C:\Users\REPLACE_ME\Desktop\images\data-10-04-2024-17-42-38.h5"
data_extra, _ = mcs.load(data_path, key="data_channels_extra")
data_laser = data_extra[:, :, :, :, :, 1]
print(data_laser.shape)
data_laser_hist = np.sum(data_laser, axis = (0,1,2,3))
phasor_laser = flim.calculate_phasor(data_laser_hist)
print(phasor_laser)

In [ ]:
#load data laser (26th channel) for the IRF and calculate the correspondent phasor

data_extra_irf, _ = mcs.load(data_path_irf, key="data_channels_extra")
data_laser_irf = data_extra_irf[:, :, :, :, :, 1]
print(data_laser_irf.shape)
data_laser_hist_irf = np.sum(data_laser_irf, axis = (0,1,2,3))
phasor_laser_irf = flim.calculate_phasor(data_laser_hist_irf)
print(phasor_laser_irf)

In [ ]:
nch = data_hist_irf.shape[-1]
print("nchannels", nch)

shift_vec = np.empty( nch )

for i in range(nch):
    shift_vec[i], *_ = phase_cross_correlation(data_hist_irf[:, 12], data_hist_irf[:, i], upsample_factor=10, normalization=None)


irf_shifted = np.empty_like(data_hist_irf)
for i in range(nch):
    irf_shifted[:, i] = shift(data_hist_irf[:, i], shift_vec[i], order = 1, mode='grid-wrap')
    
print("irf_shifted", irf_shifted.shape)
plt.figure()
plt.plot(irf_shifted)


dset_shifted = np.empty_like(data)
shift_dset = np.zeros( (data.ndim - 1, nch) )
shift_dset[-1] = shift_vec
print("shift_dset", shift_dset.shape)

for i in range(nch):
    dset_shifted[..., i] = shift(data[..., i], shift_dset[:, i], order = 1, mode='grid-wrap')
    
print("dset_shifted", dset_shifted.shape)

data_3D = dset_shifted.sum(-1) 

###  Save in a .h5 file the phasors computed from the pixels'decay histograms performing a FFT on the histograms' time bins and extracting the first harmonic

In [ ]:
flim.phasor_h5(data_path = r"C:\Users\REPLACE_ME\Desktop\images\data-10-04-2024-17-42-38.h5", data_input = data_3D) 

### Pixels' phasors are displayed in the phasor plot. Fluorescene decay histograms are not aligned with the first time bin and have not been corrected for the IRF . 


In [ ]:
%matplotlib widget

hf_phasors_per_pixel = h5py.File(r"C:\Users\REPLACE_ME\Desktop\images\data-10-04-2024-17-42-38_phasors_matrix.h5", "r")
print(hf_phasors_per_pixel.keys())

phasors_pix = hf_phasors_per_pixel["h5_dataset_phasor_pix"]  # data with phasors in each pixel

print(phasors_pix.shape)

flim.plot_phasor(phasors_pix[:], bins_2dplot=200, log_scale=False, quadrant='all')

### Calculate phasor of the irf and the correction factor from the laser's (26th channel's) phasor to realign the phasors of the data in the phasor plot

In [ ]:
irf_summed = np.sum(irf_shifted, axis = -1)
print(irf_summed.shape)
phasor_ir = flim.phasor(irf_summed)
print(phasor_ir)

In [ ]:
#  histograms
corr = flim.correction_phasor(data_laser_hist, data_laser_hist_irf)
print(corr)

In [ ]:
phasor_corrected = phasors_pix * corr / phasor_ir

### Display the phasor plot of the data realigned 

In [ ]:
flim.plot_phasor(phasor_corrected, bins_2dplot=200, log_scale=False, quadrant='first')

# Lifetime analysis

#### Calculate the fluorescence lifetime from the phasor for each pixel with the formula below (f = dfd frequency or laser rep rate frequency):
#### τ<sub>φ</sub> = (1/(2*π*f)) * tan(φ)
#### φ = arctan(s/g)
#### g = Re{phasor_corrected}
#### s = Im{phasor_corrected}


In [ ]:
tau_phi = flim.calculate_tau_phi(np.real(phasor_corrected), np.imag(phasor_corrected))
print(tau_phi.shape)



#### Calculate the fluorescence lifetime from the phasor for each pixel with the formula below (f = dfd frequency or laser rep rate frequency):
#### τ<sub>m</sub> = (1/2*π*f) * √(1/m<sup>2</sup> - 1)
#### m = √g<sup>2</sup> + s<sup>2</sup>
#### g = Re{phasor_corrected}
#### s = Im{phasor_corrected}



In [ ]:
tau_m = flim.calculate_tau_m(np.real(phasor_corrected), np.imag(phasor_corrected))
print(tau_m.shape)


### Visualize histograms of tau distribution in the pixels and intensity map of the image 

In [ ]:
tau_data = 1e9*tau_phi.flatten()

plt.figure()
plt.hist(tau_data, range = (-6, 6), bins = 50)

In [ ]:
tau_m_data = 1e9*tau_m.flatten()

plt.figure()
plt.hist(tau_m_data, range = (0, 13), bins = 50)

In [ ]:
data_histograms = np.sum(data_3D, axis = -1)
print(data_histograms.shape)
    
# Plot the histogram of the photon counts in each pixel to see the distribution (e.g. check the level of noise) 
plt.figure()
plt.hist(data_histograms.flatten(), bins = 100, range = (0, 500))

### Display and save the FLIM image representing the lifetime and intensity with a 2D colormap

In [ ]:
fig = plt.figure(figsize = (9, 6))
gs = fig.add_gridspec(4, 4)
ax1 = fig.add_subplot(gs[0:2, 0:2])
ax2 = fig.add_subplot(gs[2:4, 0:2])
flim.plot_phasor(phasor_corrected[:], bins_2dplot=200, log_scale=False, quadrant='first', fig = fig, ax = ax1)
gr.show_flim(data_histograms, tau_m*1e9, pxsize = 0.05, pxdwelltime = 324, lifetime_bounds = (1.3, 4), fig = fig, ax = ax2)  
fig.tight_layout()
plt.savefig(r"C:\Users\REPLACE_ME\Desktop\PDF_processed_images\convallaria_C.pdf", dpi = 900)

### Plot the intensity images per channel

In [ ]:
dataset = data.sum(axis = -2)
print(dataset.shape)

In [ ]:
fig = gr.ShowDataset(dataset)

### Plot intensity image (Nx x Ny)

In [ ]:
gr.ShowImg(data_histograms, pxsize_x = 0.05)